In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report, confusion_matrix

# 1. Load Dataset
df = pd.read_csv("ObesityDataSet_raw_and_data_sinthetic.csv")

# 2. Construct Binary Target (1 = Obese, 0 = Not Obese)
obese_categories = ['Obesity_Type_I', 'Obesity_Type_II', 'Obesity_Type_III']
df['Is_Obese'] = df['NObeyesdad'].apply(lambda x: 1 if x in obese_categories else 0)

# 3. Feature Matrix (X) and Binary Target Array (y)
X = df.drop(columns=['NObeyesdad', 'Is_Obese'])
y = df['Is_Obese'].values

# 4. Preprocessing Pipeline
#numeric cols are filtered in a list 
num_cols = X.select_dtypes(include=['float64', 'int64']).columns.tolist()
#textual cols are filtered in a list 
cat_cols = X.select_dtypes(include=['object']).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), cat_cols)
    ]
)
X_processed = preprocessor.fit_transform(X)

# 5. Train-Test Split (80/20 Stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y, test_size=0.2, random_state=42, stratify=y
)

# 6. Explicitly Define ANN Architecture
model = Sequential([
    Input(shape=(X_train.shape[1],)),              # Input Layer (all preprocessed features)
    Dense(64, activation='relu'),                  # Hidden Layer 1: 64 Neurons + ReLU
    Dropout(0.2),                                  # Regularization: 20% Dropout
    Dense(32, activation='relu'),                  # Hidden Layer 2: 32 Neurons + ReLU
    Dense(1, activation='sigmoid')                 # Output Layer: 1 Neuron + Sigmoid (Probability 0 to 1)
])

# Display Layer Architecture
model.summary()

# 7. Compile Model
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# 8. Set Up Early Stopping Callback
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

# 9. Train with Explicit Epochs and Batch Size
print("\n--- Training ANN ---")
history = model.fit(
    X_train, 
    y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.15,
    callbacks=[early_stop],
    verbose=1
)

# 10. Evaluate on Test Set
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"\nFinal Test Loss: {test_loss:.4f}")
print(f"Final Test Accuracy: {test_accuracy * 100:.2f}%\n")

# 11. Generate Predictions & Classification Report
y_pred_probs = model.predict(X_test)
y_pred = (y_pred_probs >= 0.5).astype(int).flatten()

print("--- Confusion Matrix ---")
print(confusion_matrix(y_test, y_pred))

print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred, target_names=['Not_Obese', 'Obese']))

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense (Dense)                        │ (None, 64)                  │           2,048 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 32)                  │           2,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 1)                   │              33 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 4,161 (16.25 KB)

 Trainable params: 4,161 (16.25 KB)

 Non-trainable params: 0 (0.00 B)


--- Training ANN ---
Epoch 1/20
45/45 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - accuracy: 0.7008 - loss: 0.6056 - val_accuracy: 0.8386 - val_loss: 0.4303
Epoch 2/20
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.8389 - loss: 0.3842 - val_accuracy: 0.9094 - val_loss: 0.2658
Epoch 3/20
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8940 - loss: 0.2603 - val_accuracy: 0.9488 - val_loss: 0.1839
Epoch 4/20
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9247 - loss: 0.1951 - val_accuracy: 0.9606 - val_loss: 0.1318
Epoch 5/20
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9435 - loss: 0.1532 - val_accuracy: 0.9724 - val_loss: 0.1029
Epoch 6/20
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9623 - loss: 0.1213 - val_accuracy: 0.9685 - val_loss: 0.0864
Epoch 7/20
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9693 - loss: 0.0972 - val_accuracy: 0.9803 - val_loss: 0.0680
Epoch 8/20
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9826 - loss: 0.0731 - val_accur